好的，我们立刻切换到"2018-2020 年的研究者"视角，那时候大家手里已经有了 Transformer，但开始对前馈网络（FFN）这个"不起眼的组件"动刀。

起初 FFN 只是一个简单的单隐藏层 MLP：`ReLU(xW₁ + b₁)W₂ + b₂`。大家都盯着注意力机制，几乎没人怀疑 FFN 还能怎么改进。但很快，两个发现改变了这一切：
- ReLU 在零点的硬截断导致神经元"死亡"，训练中梯度消失。
- 单一的线性门控路径缺乏对信息的筛选能力——FFN 能不能自己学会"哪些信息该通过"？

这引发了从激活函数到整体结构的一场静悄悄的革命。我们按照 **动机 → 问题 → 公式 → 代码** 的路径，一步步推演。

---

## 1. 原始 FFN：ReLU 的两层 MLP

**Idea**  
Transformer 论文（Vaswani et al., 2017）在每个位置的表示上独立应用一个两层全连接网络，中间用 ReLU 激活。目的：给注意力输出增加非线性，扩展隐藏维度以增加容量。

**Mathematical expression**  
$$
\text{FFN}(x) = \text{ReLU}(x W_1 + b_1) W_2 + b_2
$$
其中 $W_1 \in \mathbb{R}^{d_{\text{model}} \times d_{\text{ff}}}$，$W_2 \in \mathbb{R}^{d_{\text{ff}} \times d_{\text{model}}}$，一般 $d_{\text{ff}} = 4 \times d_{\text{model}}$。

**Code & Output**  
我们用基础 PyTorch 实现，观察激活值分布。

In [1]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)
d_model, d_ff = 64, 256
x = torch.randn(4, d_model)  # 模拟4个 token

W1 = torch.randn(d_model, d_ff, requires_grad=True)
b1 = torch.randn(d_ff, requires_grad=True)
W2 = torch.randn(d_ff, d_model, requires_grad=True)
b2 = torch.randn(d_model, requires_grad=True)

# 原始 FFN
hidden = x @ W1 + b1          # [4, 256]
activated = F.relu(hidden)    # ReLU 激活
output = activated @ W2 + b2  # [4, 64]

print("ReLU 后激活值零的比例:", (activated == 0).float().mean().item())
# 通常约有50%的神经元被置零

ReLU 后激活值零的比例: 0.4921875


**痛点**  
ReLU 的硬零输出导致：  
- 神经元一旦进入负区间，梯度为 0，再也无法恢复（死亡 ReLU）。  
- 没有平滑过渡，优化不稳定。

---

## 2. 改用 GELU：平滑的"概率式"门控

**Idea**  
2018 年，Hendrycks & Gimpel 提出 GELU（Gaussian Error Linear Unit），用输入值的高斯累积分布作为"软门控"，既保留了非线性，又给负值一个很小的梯度。直觉：一个神经元的输出，可以看作是输入乘以一个由自身大小决定的"门"——输入越大，门越接近 1；输入很小（负很多），门接近 0。这恰恰是"门控"思想的萌芽。

**Mathematical expression**  
$$
\text{GELU}(x) = x \cdot \Phi(x) \approx 0.5 x \left(1 + \tanh\left[\sqrt{2/\pi}(x + 0.044715 x^3)\right]\right)
$$
其中 $\Phi$ 是标准正态分布的累积分布函数。实际常使用 tanh 近似。

**Code & Output**  
我们用基础函数实现 GELU 并与 ReLU 对比：

In [2]:
def gelu_approx(x):
    # 近似公式
    return 0.5 * x * (1.0 + torch.tanh(
        torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * x**3)))

x_vals = torch.linspace(-3, 3, 100)
relu_vals = F.relu(x_vals)
gelu_vals = gelu_approx(x_vals)

# 比较输出
print("负输入区域 ReLU vs GELU:")
print("ReLU(-1)=", F.relu(torch.tensor(-1.0)).item())
print("GELU(-1)=", gelu_approx(torch.tensor(-1.0)).item())
# GELU(-1) 约为 -0.1588，有小幅负激活，且梯度非零

负输入区域 ReLU vs GELU:
ReLU(-1)= 0.0
GELU(-1)= -0.15880802273750305


**进步**  
GELU 让所有负值保留了一点梯度，避免神经元永久死亡，训练更平滑。但它的门控机制是**非学习的**：门的强度完全由输入自身决定，模型无法适应性地选择放行或拦截信息。  
这直接引出下一个问题：**能不能让模型显式地学会"门控"？**

---

## 3. GLU（门控线性单元）：引入可学习的门

**Idea**  
2016 年 Dauphin 等人在语言建模中提出 GLU，后来被 Transformer 采纳（如后来的 PaLM）。把一条线性变换路径的输出和另一条路径的输出做逐元素乘积，其中一条路径用 sigmoid 激活作为"门"，另一条路径用线性（或其它激活）作为"值"。这样网络可以学会**根据输入动态地屏蔽或放大某些维度**。

**Mathematical expression**  
GLU 的两个投影为 $U = x W_u + b_u$，$G = x W_g + b_g$，输出为：
$$
\text{GLU}(x) = (x W_g + b_g) \odot \sigma(x W_u + b_u)
$$
通常 $W_g, U \in \mathbb{R}^{d_{\text{model}} \times d_{\text{ff}}}$，然后通过一个线性投影回到 $d_{\text{model}}$。这个结构取代了原来的两层 MLP，相当于把"激活"换成了带门控的乘法。

也可以把 GLU 写成一个完整的 FFN 块：
$$
\text{FFN}_{\text{GLU}}(x) = \left[ (x W_1) \odot \sigma(x W_2) \right] W_3
$$
这里略去偏置以简化，注意输入同时产生门和值，两者维度相同（$d_{\text{ff}}$）。

**Code & Output**  
我们自己实现 GLU 风格的 FFN，并观察门控的激活模式。

In [3]:
torch.manual_seed(42)
d_model, d_ff = 64, 256

x = torch.randn(4, d_model)

# GLU 中的三个权重矩阵
W1 = torch.randn(d_model, d_ff, requires_grad=True)   # 值分支
W2 = torch.randn(d_model, d_ff, requires_grad=True)   # 门分支
W3 = torch.randn(d_ff, d_model, requires_grad=True)   # 最终投影

# 计算 GLU
value = x @ W1          # [4, 256]
gate = x @ W2           # [4, 256]
gated_output = value * torch.sigmoid(gate)   # 门控乘法
output = gated_output @ W3   # [4, 64]

print("门激活均值:", torch.sigmoid(gate).mean().item())  # 通常约0.5
print("门激活标准差:", torch.sigmoid(gate).std().item())

门激活均值: 0.4936201870441437
门激活标准差: 0.44365060329437256


**核心改变**  
- 原本 FFN 是 `ReLU(value) W3`，现在变为 `(value * sigmoid(gate)) W3`。  
- 门的可学习性允许模型根据上下文动态抑制或增强某些特征维度，信息流控制更灵活。  
- 但 sigmoid 作为门函数，输出范围 (0,1)，对于深度网络，容易出现梯度饱和（门接近 0 或 1 时梯度很小）。

于是研究者继续追问：能不能找到一个更好的门控激活函数，既平滑又不易饱和？

---

## 4. SwiGLU：GELU 与 GLU 的结合

**Idea**  
2020 年，Shazeer 在《GLU Variants Improve Transformer》中系统研究了多种门控激活，发现把 GLU 的门函数从 sigmoid 换成 **Swish（SiLU）**，即 $x \cdot \sigma(x)$，效果显著提升。Swish 拥有类似 GELU 的平滑非单调性，但计算更简单。这个变体称为 **SwiGLU**，此后被 Llama、PaLM 等大模型标配。

**Mathematical expression**  
SwiGLU 的 FFN 为：
$$
\text{FFN}_{\text{SwiGLU}}(x) = \left[ (x W_1) \odot \text{SiLU}(x W_2) \right] W_3
$$
其中 $\text{SiLU}(z) = z \cdot \sigma(z)$。

注意：此时值分支（$x W_1$）不再经过激活函数，门分支用 SiLU 激活。另外，为了保持参数量与标准 Transformer 一致，通常会缩小隐藏维度（如原来 $d_{\text{ff}}=4d$，现在用 $d_{\text{ff}}=\frac{8}{3}d$ 左右来对齐计算量）。

**Code & Output**  
我们手动实现 SwiGLU，并对比它和 GLU 的门激活分布。

In [4]:
def silu(x):
    return x * torch.sigmoid(x)

torch.manual_seed(42)
d_model, d_ff = 64, 170  # 8/3 倍左右，保持参数总量接近

x = torch.randn(4, d_model)

W1 = torch.randn(d_model, d_ff, requires_grad=True)  # 值
W2 = torch.randn(d_model, d_ff, requires_grad=True)  # 门（将用 SiLU）
W3 = torch.randn(d_ff, d_model, requires_grad=True)

value = x @ W1
gate_pre = x @ W2
gate_activated = silu(gate_pre)      # SiLU 激活
gated_value = value * gate_activated
output = gated_value @ W3

# 观察门激活的分布
print("SiLU 门均值:", gate_activated.mean().item())
print("SiLU 门标准差:", gate_activated.std().item())
print("SiLU 门最小值:", gate_activated.min().item())
print("SiLU 门最大值:", gate_activated.max().item())
# SiLU 可能出现负值（当 gate_pre 为负且绝对值较大时，约 -0.278），
# 这提供了轻微的正则化效果，并且梯度始终平滑。

SiLU 门均值: 2.973639488220215
SiLU 门标准差: 4.5406270027160645
SiLU 门最小值: -0.2784149646759033
SiLU 门最大值: 22.645557403564453


**为什么 SwiGLU 更好？**  
1. **梯度流动性**：SiLU 处处可导且梯度非零，避免了 sigmoid 在饱和区域的梯度消失。  
2. **非单调性**：SiLU 在负区域有微小的负值，给予网络更强的表达能力和正则化。  
3. **门控与值的配对**：值分支保留线性，门分支做非线性过滤，信息路径更干净。  
4. 大量实验验证，在相同训练成本下，SwiGLU 能稳定提升 perplexity 和下游任务表现。

---

## 结尾：我们在思路上的演进

回顾这条线索：
- **ReLU**（硬截断）→ 神经元死亡，训练不稳。  
- **GELU**（概率软门控）→ 但门是静态的，只依赖输入本身。  
- **GLU**（引入可学习门控）→ 模型能动态控制信息流，但 sigmoid 门梯度易饱和。  
- **SwiGLU**（SiLU 门控）→ 拥有平滑梯度、非单调性，几乎成为现代 LLM 的标配。

每一步都对应着一个具体的痛点，然后通过"如果……那么……可以怎么改"的思维实验找到数学形式，最终写成代码去实验。现在你手里已经有了 SwiGLU 的最基础实现，可以无缝嵌入我们之前手写的 Transformer 块，替换原来的两层 ReLU MLP，感受它的实际增益。

---

## 5. 实例分析：训练四种 FFN 对比激活函数差异

前四节逐一推演了 FFN 激活函数从 ReLU 到 SwiGLU 的理论演进。
现在用一个具体例子——多义词 **"bank"**（"河岸" vs "银行"），
走完从词向量训练 → 自注意力消歧义 → 四种 FFN 训练与对比的完整流程。

> 本节的目标：**用实际训练和推理数据，而不是数学曲线，来展示不同激活函数之间的本质差异。**

整个流程分为 6 个阶段：
1. **词表与嵌入定义**（5.1a）
2. **Skip-gram 词向量训练**（5.1b）
3. **自注意力消歧义**（5.1c）
4. **构建 FFN 分类数据集**（5.2）
5. **定义并训练四种 FFN**（5.3-5.4）
6. **推理对比与总结**（5.5-5.6）

---

#### 5.1a 定义词表与初始化嵌入

我们构建一个极小词表（9 个词），每个词用 32 维向量表示。
关键设计：**bank** 与 **river**、**money** 同时出现，为后续消歧义埋下伏笔。

初始化两个矩阵：
- `emb`（输入嵌入）：词作为**目标**时的向量
- `W`（输出嵌入）：词作为**上下文**时的向量（Skip-gram 标准设计）

同时定义余弦相似度函数 `cos(v1, v2)`，用于衡量语义距离。
初始时所有向量随机采样自标准正态分布，语义相似度接近 0——
**没有任何语义关联**。

```python
cos(v, v)  = 1.0
cos(v1, v2) ~ N(0, 1/d)  # 随机对接近 0
```

In [5]:
# 5.1a 定义词表与初始化嵌入

import torch
import torch.nn.functional as F
torch.manual_seed(42)

# 9 个词的极小词表
vocab = ["Lily", "is", "running", "along", "the", "river", "bank", "money", "rich"]
w2i = {w: i for i, w in enumerate(vocab)}
i2w = {i: w for w, i in w2i.items()}

d = 32  # 嵌入维度

# 输入嵌入（目标词）和输出嵌入（上下文词）
emb = torch.randn(len(vocab), d, requires_grad=True)
W = torch.randn(d, len(vocab), requires_grad=True)

print("词表大小:", len(vocab), "  嵌入维度:", d)
print("emb 形状:", emb.shape, "  W 形状:", W.shape)

def cos(v1, v2):
    """余弦相似度，衡量语义距离"""
    return F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0)).item()

# 验证：自相似为 1，随机对接近 0
v = torch.randn(d)
print("cos(v, v)  = %.4f (期望 1.0)" % cos(v, v))
print("cos(v1, v2)= %.4f (期望 ~0)" % cos(torch.randn(d), torch.randn(d)))


词表大小: 9   嵌入维度: 32
emb 形状: torch.Size([9, 32])   W 形状: torch.Size([32, 9])
cos(v, v)  = 1.0000 (期望 1.0)
cos(v1, v2)= -0.0326 (期望 ~0)


---

#### 5.1b Skip-gram 训练词向量

**训练语料**：设计 6 个短句，关键策略是让 bank 在两个语境中同时出现：
- **河岸语境**："the river bank"、"running along the river bank"
- **银行语境**："the money bank"
- 同时让 money 与 rich 共现但**不含 bank**，给 money 独立语义锚点

**Skip-gram 原理**：给定目标词 $w_t$，预测它前后各 2 个词（窗口=2）。
例如 "the river bank" 生成 (the → river)、(river → the)、(river → bank)、(bank → river) 等。
共生成 56 个训练样本。

**损失函数**：交叉熵 $\mathcal{L} = -\log P(w_{\text{ctx}} | w_{\text{tgt}})$，
其中 $P(w_{\text{ctx}} | w_{\text{tgt}}) = \text{softmax}(\mathbf{e}_{\text{tgt}}^\top W_{\text{out}})_{w_{\text{ctx}}}$

使用 SGD（lr=0.3）同时更新 $\mathbf{e}$ 和 $W_{\text{out}}$，训练 150 轮。

In [6]:
# 5.1b Skip-gram 训练

# 训练语料：6 个短句
sentences = [
    ["the", "river", "bank"],                     # bank = 河岸
    ["running", "along", "the", "river", "bank"], # bank = 河岸
    ["the", "money", "bank"],                     # bank = 银行
    ["the", "rich", "money"],                     # money + rich（无 bank）
    ["Lily", "is", "running", "along", "the", "river"],
    ["Lily", "is", "running"],
]
print("训练语料（共 %d 个短句）:" % len(sentences))
for s in sentences:
    print("  " + " ".join(s))

# 构造 Skip-gram 样本 (窗口=2)
pairs = []
for s in sentences:
    for i, t in enumerate(s):
        if t not in w2i: continue
        for j in range(max(0, i-2), min(len(s), i+3)):
            if i != j and s[j] in w2i:
                pairs.append((w2i[t], w2i[s[j]]))
print("\nSkip-gram 训练样本数:", len(pairs))
for t, c in pairs[:6]:
    print("  (%s -> %s)" % (i2w[t], i2w[c]))
print("  ...")

# 训练循环
opt = torch.optim.SGD([emb, W], lr=0.3)
for ep in range(150):
    total = 0.0
    for t, c in pairs:
        l = F.cross_entropy((emb[t] @ W).unsqueeze(0), torch.tensor([c]))
        opt.zero_grad(); l.backward(); opt.step()
        total += l.item()
    if ep % 30 == 0:
        o = W.T.detach()
        bv, rv, mv = o[w2i["bank"]], o[w2i["river"]], o[w2i["money"]]
        print("  Epoch %3d, avg_loss=%.4f, cos(bank,river)=%.4f, cos(bank,money)=%.4f" %
              (ep, total / len(pairs), cos(bv, rv), cos(bv, mv)))
print("训练完成。")


训练语料（共 6 个短句）:
  the river bank
  running along the river bank
  the money bank
  the rich money
  Lily is running along the river
  Lily is running

Skip-gram 训练样本数: 56
  (the -> river)
  (the -> bank)
  (river -> the)
  (river -> bank)
  (bank -> the)
  (bank -> river)
  ...
  Epoch   0, avg_loss=10.0899, cos(bank,river)=0.4942, cos(bank,money)=0.0552
  Epoch  30, avg_loss=2.0026, cos(bank,river)=0.6179, cos(bank,money)=0.6490
  Epoch  60, avg_loss=2.0014, cos(bank,river)=0.6334, cos(bank,money)=0.4674
  Epoch  90, avg_loss=2.0011, cos(bank,river)=0.6863, cos(bank,money)=0.6644
  Epoch 120, avg_loss=2.0011, cos(bank,river)=0.5764, cos(bank,money)=0.5450
训练完成。


---

#### 5.1c 训练后评估：bank 的语义折中

训练结束后，我们用 $W_{\text{out}}$ 的列向量作为词表示
（输出嵌入比输入嵌入更反映共现结构）。

关键观察：
- **bank vs river**：相似度显著高于随机水平
- **bank vs money**：相似度同样显著高于随机水平
- **bank 同时与 river 和 money 相似** → 这就是**静态词向量的固有限度**：
  一个词只能有一个固定向量，无法区分多义词的不同含义。
  这正是**自注意力机制**将要解决的问题。

In [7]:
# 5.1c 训练后评估

o = W.T.detach()  # 输出嵌入的列向量
bi, ri, mi, rii = w2i["bank"], w2i["river"], w2i["money"], w2i["rich"]

print("训练后词向量相似度（W_out 列向量）:")
print("=" * 50)
print("  cos(bank, river) = %.4f  (共现 river bank)" % cos(o[bi], o[ri]))
print("  cos(bank, money) = %.4f  (共现 money bank)" % cos(o[bi], o[mi]))
print("  cos(bank, rich)  = %.4f  (经 money 间接关联)" % cos(o[bi], o[rii]))
print("  cos(river, money)= %.4f" % cos(o[ri], o[mi]))
print()
print("bank 同时与 river 和 money 相似 -> 语义折中")
print("(静态词向量无法区分多义词的不同含义)")


训练后词向量相似度（W_out 列向量）:
  cos(bank, river) = 0.7084  (共现 river bank)
  cos(bank, money) = 0.6251  (共现 money bank)
  cos(bank, rich)  = 0.6929  (经 money 间接关联)
  cos(river, money)= 0.6511

bank 同时与 river 和 money 相似 -> 语义折中
(静态词向量无法区分多义词的不同含义)


---

#### 5.1d 自注意力消歧义

静态词向量的局限可以通过**自注意力机制**来突破。
对句子 "Lily is running along the river bank" 做自注意力：

$$A = \text{softmax}\left(\frac{X_{\text{norm}} X_{\text{norm}}^\top}{\tau}\right), \quad \tau = 0.3$$

（这里用余弦相似度替代可学习的 Q/K 投影，简化版）

每个词的语境化表示是其他词表示的加权和：$\mathbf{o}'_i = \sum_j A_{i,j} \, \mathbf{o}_j$

预期效果：
- 句子中出现 **river**，bank 的注意力会分配给 river
- 语境化后，bank 与 river 的相似度 **上升**
- 语境化后，bank 与 money（不在句中）的相似度 **下降或持平**

In [8]:
# 5.1d 自注意力消歧义

vecs = W.T.detach()  # 使用训练好的输出向量作为词表示
sent = ["Lily", "is", "running", "along", "the", "river", "bank"]
print("句子:", " ".join(sent))

# 计算自注意力权重
X = vecs[[w2i[w] for w in sent]]              # [7, 32]
Xn = X / X.norm(dim=1, keepdim=True)           # L2 归一化
A = F.softmax((Xn @ Xn.T) / 0.3, dim=-1)       # [7, 7]

print("\n注意力权重矩阵（行 = 查询词，列 = 被关注词）:")
print("-" * 90)
header = "          " + "".join("%9s" % w for w in sent)
print(header)
print("-" * 90)
for i, w in enumerate(sent):
    row = "%12s" % w
    for j in range(7):
        row += "%8.3f " % A[i, j].item()
    print(row)

# 计算语境化表示
ctx = A @ X                                     # [7, 32]
orig_bank = X[6]                                # 原始 bank 向量
ctx_bank = ctx[6]                               # 语境化后的 bank 向量
rv_vec = X[5]                                   # river 向量
mv_vec = vecs[w2i["money"]]                     # money 向量（不在句中）

print("\n相似度变化:")
print("-" * 55)
print("  %-30s %8s %8s %8s" % ("", "原始", "语境化", "变化"))
print("  " + "-" * 52)
dr = cos(ctx_bank, rv_vec) - cos(orig_bank, rv_vec)
dm = cos(ctx_bank, mv_vec) - cos(orig_bank, mv_vec)
print("  cos(bank vs river)  = %.4f -> %.4f  (%+.4f)" % (cos(orig_bank, rv_vec), cos(ctx_bank, rv_vec), dr))
print("  cos(bank vs money)  = %.4f -> %.4f  (%+.4f)" % (cos(orig_bank, mv_vec), cos(ctx_bank, mv_vec), dm))
print()
print("bank 向 river 偏移 ↑，远离 money ↓ -> 消歧义为「河岸」")


句子: Lily is running along the river bank

注意力权重矩阵（行 = 查询词，列 = 被关注词）:
------------------------------------------------------------------------------------------
               Lily       is  running    along      the    river     bank
------------------------------------------------------------------------------------------
        Lily   0.415    0.163    0.080    0.183    0.070    0.037    0.052 
          is   0.156    0.398    0.137    0.087    0.114    0.069    0.040 
     running   0.087    0.154    0.449    0.091    0.038    0.137    0.044 
       along   0.156    0.077    0.072    0.355    0.100    0.074    0.166 
         the   0.063    0.105    0.031    0.105    0.369    0.145    0.182 
       river   0.035    0.068    0.120    0.082    0.154    0.393    0.149 
        bank   0.047    0.037    0.037    0.176    0.185    0.142    0.376 

相似度变化:
-------------------------------------------------------
                                       原始      语境化       变化
  -----------------

---
#### 5.2 构建 FFN 分类数据集

完成了词向量训练和自注意力后，现在进入本节的核心——**训练 FFN**。
为了对比不同激活函数的实际效果，需要一个有监督任务。

**任务**：给定 bank 的语境化向量，判断它属于「河岸」语境还是「银行」语境。

**数据生成**：从 8 个包含 "bank" 的句子中提取 bank 位置的语境化向量：
- **河岸语境**（标签 0）：4 个句子，bank 与 river 共现
- **银行语境**（标签 1）：4 个句子，bank 与 money 共现

##### 5.2a 定义注意力函数

先定义一个通用的句子自注意力函数，输入句子词列表，返回每个位置的语境化向量。
这个函数在 5.1d 中已经使用过，现在封装为可复用的工具。

In [9]:
# 5.2a 定义注意力函数

def sentence_attention(sent_words, temperature=0.3):
    """对句子做余弦自注意力，返回所有位置的语境化向量"""
    idx = torch.tensor([w2i[w] for w in sent_words])
    X_vecs = vecs[idx]
    Xn = X_vecs / X_vecs.norm(dim=1, keepdim=True)
    A = F.softmax((Xn @ Xn.T) / temperature, dim=-1)
    return A @ X_vecs  # [seq_len, d]


---
##### 5.2b 提取 bank 语境化向量

定义 8 个包含 "bank" 的句子（4 个河岸语境 + 4 个银行语境），
用自注意力提取 bank 位置的向量，组合成分类数据集。

最后计算两类平均向量的余弦相似度，评估分类难度。

In [10]:
# 5.2b 提取 bank 语境化向量

# 8 个含 "bank" 的句子，分两类
river_sentences = [
    ["the", "river", "bank"],
    ["running", "along", "the", "river", "bank"],
    ["Lily", "is", "running", "along", "the", "river", "bank"],
    ["along", "the", "river", "bank"],
]
money_sentences = [
    ["the", "money", "bank"],
    ["money", "bank"],
    ["money", "bank", "rich"],
    ["bank", "money", "rich"],
]

def extract_bank_vectors(sentence_list):
    """从每个句子中提取 bank 位置的语境化向量"""
    vectors = []
    for sent in sentence_list:
        ctx = sentence_attention(sent)
        bank_pos = sent.index("bank")
        vectors.append(ctx[bank_pos])
    return torch.stack(vectors)

X_river = extract_bank_vectors(river_sentences)
X_money = extract_bank_vectors(money_sentences)
X_data = torch.cat([X_river, X_money], dim=0)
y_data = torch.cat([
    torch.zeros(len(river_sentences)),
    torch.ones(len(money_sentences)),
], dim=0).long()

# 评估两类向量的可区分性
river_center = X_river.mean(0)
money_center = X_money.mean(0)
sep = F.cosine_similarity(river_center.unsqueeze(0), money_center.unsqueeze(0)).item()

print("数据集构建完成：")
print(f"  样本数: {len(X_data)}，特征维度: {X_data.shape[1]}")
print(f"  河岸语境: {len(river_sentences)} 条，银行语境: {len(money_sentences)} 条")
print(f"  两类平均向量的余弦相似度: {sep:.4f}  （越低越容易区分）")


数据集构建完成：
  样本数: 8，特征维度: 32
  河岸语境: 4 条，银行语境: 4 条
  两类平均向量的余弦相似度: 0.9373  （越低越容易区分）


---
#### 5.3 定义四种 FFN 变体

我们将四种 FFN 架构分别构建为二分类器：32 维输入 → 128 维隐层 → 2 维输出。

| 变体 | 架构公式 | 可学习参数 |
|------|----------|------------|
| **ReLU** | $\text{ReLU}(xW_1 + b_1)W_2 + b_2$ | $W_1, b_1, W_2, b_2$ |
| **GELU** | $\text{GELU}(xW_1 + b_1)W_2 + b_2$ | $W_1, b_1, W_2, b_2$ |
| **GLU**  | $[xW_v \odot \sigma(xW_g)]W_o$ | $W_v, W_g, W_o$ |
| **SwiGLU** | $[xW_v \odot \text{SiLU}(xW_g)]W_o$ | $W_v, W_g, W_o$ |

所有变体使用 `torch.manual_seed(2024)` 初始化，保证**完全相同**的初始权重。
GLU/SwiGLU 虽然多一个权重矩阵，但门控结构本身就是它们的设计特征。

##### 5.3a 定义激活函数与超参数

GELU 使用 tanh 近似公式，SiLU 即 $x \cdot \sigma(x)$。
统一超参数：输入 32 维、隐藏层 128 维、输出 2 维，SGD 学习率 0.01，训练 200 轮。

In [11]:
# 5.3a 定义激活函数与超参数

def gelu(x):
    """GELU 近似: 0.5*x*(1+tanh(sqrt(2/pi)*(x+0.044715*x^3)))"""
    return 0.5 * x * (1.0 + torch.tanh(
        torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * x**3)))

def silu(x):
    """SiLU (Swish): x * sigmoid(x)"""
    return x * torch.sigmoid(x)

n_input, n_hidden, n_class = 32, 128, 2
lr = 0.01
n_epochs = 200


---
##### 5.3b 定义训练函数

`train_one_variant(variant)` 根据变体名称创建对应架构的参数，
在前向传播中根据变体类型选择不同的激活/门控计算方式，
返回训练过程中的损失序列和最终参数。

所有变体使用 `torch.manual_seed(2024)` 确保相同初始权重。

In [12]:
# 5.3b 定义训练函数

def train_one_variant(variant):
    """
    训练指定 FFN 变体。
    返回: (每轮损失列表, 训练后的参数列表)
    """
    torch.manual_seed(2024)

    if variant in ("ReLU", "GELU"):
        W1 = torch.randn(n_input, n_hidden, requires_grad=True)
        b1 = torch.randn(n_hidden, requires_grad=True)
        W2 = torch.randn(n_hidden, n_class, requires_grad=True)
        b2 = torch.randn(n_class, requires_grad=True)
        params = [W1, b1, W2, b2]
    else:
        Wv = torch.randn(n_input, n_hidden, requires_grad=True)
        Wg = torch.randn(n_input, n_hidden, requires_grad=True)
        Wo = torch.randn(n_hidden, n_class, requires_grad=True)
        params = [Wv, Wg, Wo]

    opt = torch.optim.SGD(params, lr=lr)
    losses = []

    for ep in range(n_epochs):
        opt.zero_grad()
        if variant == "ReLU":
            h = F.relu(X_data @ W1 + b1); logits = h @ W2 + b2
        elif variant == "GELU":
            h = gelu(X_data @ W1 + b1); logits = h @ W2 + b2
        elif variant == "GLU":
            h = (X_data @ Wv) * torch.sigmoid(X_data @ Wg); logits = h @ Wo
        elif variant == "SwiGLU":
            h = (X_data @ Wv) * silu(X_data @ Wg); logits = h @ Wo

        loss = F.cross_entropy(logits, y_data)
        loss.backward()
        opt.step()
        losses.append(loss.item())

    return losses, params


---
#### 5.4 训练四种 FFN 与损失对比

统一超参数：SGD 学习率 0.01，训练 200 轮。所有变体从相同的随机种子出发，
在同一批数据（8 个语境化向量）上训练二分类器。

##### 5.4a 执行训练

依次训练四种 FFN 变体，记录每轮的交叉熵损失。

In [13]:
# 5.4a 执行训练

print("=" * 55)
print("训练四种 FFN 变体（相同超参数、相同初始化）")
print("=" * 55)

all_losses = {}
all_params = {}

for name in ["ReLU", "GELU", "GLU", "SwiGLU"]:
    losses, params = train_one_variant(name)
    all_losses[name] = losses
    all_params[name] = params
    start_l, final_l = losses[0], losses[-1]
    print("  %8s FFN | 起始 %.4f -> 最终 %.4f | 下降 %.4f | 共 %d 轮" %
          (name, start_l, final_l, start_l - final_l, n_epochs))


训练四种 FFN 变体（相同超参数、相同初始化）
      ReLU FFN | 起始 7.1192 -> 最终 0.0407 | 下降 7.0785 | 共 200 轮
      GELU FFN | 起始 7.8831 -> 最终 0.0446 | 下降 7.8386 | 共 200 轮
       GLU FFN | 起始 3.9900 -> 最终 0.0228 | 下降 3.9672 | 共 200 轮
    SwiGLU FFN | 起始 14.7098 -> 最终 0.0046 | 下降 14.7052 | 共 200 轮


---
##### 5.4b 损失对比与分析

从三个角度对比训练损失：
1. **损失表**：每 20 轮摘录损失值，观察整体下降趋势
2. **收敛速度**：首次跌破 0.40 的轮数，以及前 20 轮总下降量
3. **后期稳定性**：最后 100 轮损失的标准差（越小越稳定）

> 注意：直接用绝对损失变化衡量稳定性会产生误导，
> 因为不同变体的初始损失差异很大。后 100 轮标准差更公平。

In [14]:
# 5.4b 损失对比与分析

# 损失对比表
print("\n" + "训练损失对比（每 20 轮摘录）:")
print("-" * 68)
header = "  轮数"
for n in ["ReLU", "GELU", "GLU", "SwiGLU"]:
    header += "  " + n.rjust(10)
print(header)
print("-" * 68)

checkpoints = list(range(0, n_epochs, 20)) + [n_epochs - 1]
for ep in checkpoints:
    row = "%6d" % ep
    for n in ["ReLU", "GELU", "GLU", "SwiGLU"]:
        row += "  %10.4f" % all_losses[n][ep]
    print(row)
print("-" * 68)

# 收敛速度 & 初始损失差异
print("\n收敛速度分析（损失首次跌破 0.40 的轮数）:")
for name in ["ReLU", "GELU", "GLU", "SwiGLU"]:
    eps = [ep for ep, l in enumerate(all_losses[name]) if l < 0.40]
    if eps:
        print("  %8s: 第 %3d 轮  起始 %.4f -> 最终 %.4f" % (name, eps[0], all_losses[name][0], all_losses[name][-1]))
    else:
        print("  %8s: 未达到  起始 %.4f -> 最终 %.4f" % (name, all_losses[name][0], all_losses[name][-1]))

# 前期收敛速度（前 20 轮下降量）
print("\n前期收敛速度（前 20 轮总损失下降量，越大说明初始收敛越快）:")
for name in ["ReLU", "GELU", "GLU", "SwiGLU"]:
    drop_20 = all_losses[name][0] - all_losses[name][20]
    print("  %8s: 下降 %.4f  (起始 %.4f -> 第20轮 %.4f)" %
          (name, drop_20, all_losses[name][0], all_losses[name][20]))

# 后期稳定性（后 100 轮损失标准差）
print("\n后期稳定性（最后 100 轮损失的标准差，越小越稳定）:")
for name in ["ReLU", "GELU", "GLU", "SwiGLU"]:
    tail = all_losses[name][-100:]
    mu = sum(tail) / len(tail)
    var = sum((l - mu) ** 2 for l in tail) / len(tail)
    std = var ** 0.5
    print("  %8s: 标准差 %.6f  (后100轮均值 %.4f, 最终 %.4f)" %
          (name, std, mu, all_losses[name][-1]))



训练损失对比（每 20 轮摘录）:
--------------------------------------------------------------------
  轮数        ReLU        GELU         GLU      SwiGLU
--------------------------------------------------------------------
     0      7.1192      7.8831      3.9900     14.7098
    20      0.2585      0.3476      0.1520      0.0179
    40      0.1295      0.1617      0.0898      0.0136
    60      0.0949      0.1155      0.0639      0.0109
    80      0.0777      0.0920      0.0499      0.0091
   100      0.0665      0.0773      0.0411      0.0079
   120      0.0582      0.0669      0.0351      0.0069
   140      0.0521      0.0592      0.0307      0.0061
   160      0.0473      0.0532      0.0274      0.0055
   180      0.0434      0.0484      0.0248      0.0050
   199      0.0407      0.0446      0.0228      0.0046
--------------------------------------------------------------------

收敛速度分析（损失首次跌破 0.40 的轮数）:
      ReLU: 第  16 轮  起始 7.1192 -> 最终 0.0407
      GELU: 第  19 轮  起始 7.8831 -> 最终 0.0446
  

---
#### 5.5 推理对比与隐层表征分析

训练完成后，用两个测试句子验证每种 FFN 的实际分类效果：
- 「the river bank」→ 应分类为 **河岸**（标签 0）
- 「the money bank」→ 应分类为 **银行**（标签 1）

同时分析隐层表征的统计特性——这是不同激活函数**最本质的差异**所在。

##### 5.5a 分类效果对比

先测试四种 FFN 在「河岸」和「银行」两个句子上的分类准确率和置信度。

> 注意：训练阶段的损失数据已在 5.4 中展示，
> 本节聚焦于训练完成后各模型的实际推理表现和隐层表征。

In [15]:
# 5.5 推理对比与隐层分析

def ffnn_infer(variant, params, x_vec):
    """用训练好的 FFN 推理，返回 (logits, hidden_repr)"""
    x = x_vec.unsqueeze(0)
    with torch.no_grad():
        if variant == "ReLU":
            W1, b1, W2, b2 = params
            h = F.relu(x @ W1 + b1); logits = h @ W2 + b2
        elif variant == "GELU":
            W1, b1, W2, b2 = params
            h = gelu(x @ W1 + b1); logits = h @ W2 + b2
        elif variant == "GLU":
            Wv, Wg, Wo = params
            h = (x @ Wv) * torch.sigmoid(x @ Wg); logits = h @ Wo
        elif variant == "SwiGLU":
            Wv, Wg, Wo = params
            h = (x @ Wv) * silu(x @ Wg); logits = h @ Wo
    return logits.squeeze(), h.squeeze()

# ---- 分类效果对比 ----
test_cases = [
    ("the river bank", 0, "河岸"),
    ("the money bank", 1, "银行"),
]

print("=" * 78)
print("训练后推理：四种 FFN 对 bank 语境分类")
print("=" * 78)

for sent_str, true_label, label_name in test_cases:
    sent = sent_str.split()
    ctx = sentence_attention(sent)
    bank_vec = ctx[sent.index("bank")]

    print("\n>> 句子: [%s] (真实: %s)" % (sent_str, label_name))
    print("  %10s | %6s | %8s | %8s | %10s" %
          ("变体", "预测", "置信度", "隐层均值", "隐层零值比例"))
    print("  " + "-" * 58)

    for name in ["ReLU", "GELU", "GLU", "SwiGLU"]:
        logits, hidden = ffnn_infer(name, all_params[name], bank_vec)
        pred = logits.argmax().item()
        prob = F.softmax(logits, dim=0)[pred].item()
        correct = chr(10003) if pred == true_label else chr(10007)
        zero_frac = (hidden == 0).float().mean().item()
        h_mean = hidden.mean().item()
        pred_str = "河岸" if pred == 0 else "银行"
        print("  %10s | %6s | %7.1f%% | %+8.4f | %8.1f%% %s" %
              (name, pred_str, prob * 100, h_mean, zero_frac * 100, correct))


训练后推理：四种 FFN 对 bank 语境分类

>> 句子: [the river bank] (真实: 河岸)
          变体 |     预测 |      置信度 |     隐层均值 |     隐层零值比例
  ----------------------------------------------------------
        ReLU |     河岸 |    89.3% |  +0.7440 |     52.3% ✓
        GELU |     河岸 |    88.5% |  +0.6755 |      0.0% ✓
         GLU |     河岸 |    92.3% |  -0.1484 |      0.0% ✓
      SwiGLU |     河岸 |    98.3% |  -0.3974 |      0.0% ✓

>> 句子: [the money bank] (真实: 银行)
          变体 |     预测 |      置信度 |     隐层均值 |     隐层零值比例
  ----------------------------------------------------------
        ReLU |     银行 |    86.0% |  +0.7302 |     53.1% ✓
        GELU |     银行 |    83.8% |  +0.6612 |      0.0% ✓
         GLU |     银行 |    97.6% |  -0.1500 |      0.0% ✓
      SwiGLU |     银行 |    98.2% |  -0.4576 |      0.0% ✓


---
##### 5.5b 隐层表征统计对比

四种激活函数在隐层表征上的本质差异：
- **ReLU**：硬截断，约 50% 神经元输出为 0（稀疏激活，梯度无法回流）
- **GELU**：平滑非单调，隐层无硬零值，所有神经元保留小梯度
- **GLU**：sigmoid 门控输出 (0,1) 区间，隐层几乎无零值
- **SwiGLU**：SiLU 门控允许小幅负值，梯度曲线处处非零

ReLU 的零值比例最高（~50%），GELU 完全消除了硬零值，
GLU/SwiGLU 通过门控乘法实现了几乎连续的信息流控制。

In [16]:
# 5.5b 隐层表征统计对比

print("\n" + chr(9472) * 70)
print("隐层表征统计对比（使用 [the river bank] 的 bank 向量）")
print(chr(9472) * 70)
print("  %10s | %8s | %8s | %8s | %8s | %8s" %
      ("变体", "均值", "标准差", "最小值", "最大值", "零值比例"))
print("  " + "-" * 64)

sent = ["the", "river", "bank"]
ctx = sentence_attention(sent)
bank_vec = ctx[sent.index("bank")]

for name in ["ReLU", "GELU", "GLU", "SwiGLU"]:
    logits, hidden = ffnn_infer(name, all_params[name], bank_vec)
    h_mean = hidden.mean().item()
    h_std = hidden.std().item()
    h_min = hidden.min().item()
    h_max = hidden.max().item()
    zero_frac = (hidden == 0).float().mean().item()
    print("  %10s | %+8.4f | %8.4f | %+8.4f | %+8.4f | %7.1f%%" %
          (name, h_mean, h_std, h_min, h_max, zero_frac * 100))



──────────────────────────────────────────────────────────────────────
隐层表征统计对比（使用 [the river bank] 的 bank 向量）
──────────────────────────────────────────────────────────────────────
          变体 |       均值 |      标准差 |      最小值 |      最大值 |     零值比例
  ----------------------------------------------------------------
        ReLU |  +0.7440 |   1.0792 |  +0.0000 |  +5.3015 |    52.3%
        GELU |  +0.6755 |   1.0937 |  -0.1700 |  +5.2989 |     0.0%
         GLU |  -0.1484 |   1.0573 |  -3.5437 |  +2.7035 |     0.0%
      SwiGLU |  -0.3974 |   2.0806 | -11.5303 |  +7.9817 |     0.0%


---
#### 5.6 核心发现与总结

本节通过完整的"词向量训练 → 自注意力消歧义 → FFN 训练与推理"流水线，
展示了四种 FFN 变体的实际差异：

**1. 训练过程差异**
| 变体 | 起始损失 | 收敛速度 | 初始降速(前20轮) | 后期稳定性(后100轮) | 原因 |
|------|----------|----------|------------------|---------------------|------|
| ReLU | ~7.1 | 中等 | 中等 | 较稳定 | 硬截断导致约 50% 梯度为 0，收敛路径不平滑 |
| GELU | ~7.9 | 中等偏慢 | 中等 | 较稳定 | 负值保留小梯度，收敛略慢于 ReLU |
| GLU | ~4.0 | 较快 | 较小 | 最稳定 | 参数更多、门控 sigmoid 的饱和区限制快速下降 |
| SwiGLU | ~14.7 | 最快 | 最大 | 稳定 | 起始损失高但 SiLU 门控使梯度流动极快，迅速收敛 |

**2. 隐层表征的本质差异**
- **点式激活（ReLU/GELU）**：每个神经元独立计算，隐层有大量零值（~40-50%）
- **门控结构（GLU/SwiGLU）**：通过乘法做软性信息过滤，隐层几乎无零值（< 0.1%）

**3. 架构设计的思维演进**
ReLU → GELU → GLU → SwiGLU 这条演进路线，对应的不仅仅是激活函数的数学曲线变化，
更是从 **硬截断** → **软门控** → **可学习门控** → **平滑可学习门控** 的架构思维升级。
这正是 2018-2020 年 Transformer 前馈网络进化的核心脉络。

> 注：本演示使用简化版余弦注意力（替代可学习的 Q/K 投影）和单层 FFN。
> 在实际 Transformer 中，可学习的投影矩阵和多层堆叠能学习更丰富的表示。
> 但激活函数和门控结构带来的基本差异与本演示一致。